# FABEMD — Fast and Adaptive Bidimensional Empirical Mode Decomposition

Based on: **Bhuiyan, S.M.A., Adhami, R.R., Khan, J.F. (2008)**  
*"A novel approach of fast and adaptive bidimensional empirical mode decomposition."*  
IEEE ICASSP 2008.

This notebook demonstrates:
1. Generating synthetic test images
2. Running FABEMD decomposition
3. Visualising all extracted BIMFs
4. Calculating Shannon entropy for every BIMF
5. Integration with `ImageProcessingPipeline`

> **Colab compatible** — works with or without GPU. Uses CuPy when available, falls back to NumPy/SciPy.

In [ ]:
# --- Colab Setup (run this cell first on Google Colab) ---
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the repo if not already present
    import os
    if not os.path.exists("pace_implementation"):
        subprocess.check_call(["git", "clone",
            "https://github.com/Madeena-software/pace_implementation.git"])
    os.chdir("pace_implementation")

    # Install dependencies (scipy is pre-installed, cupy needs GPU runtime)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "opencv-python-headless", "scipy", "matplotlib"])
    try:
        import cupy
        print("✓ CuPy available — using GPU acceleration")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "cupy-cuda12x"])
        try:
            import cupy
            print("✓ CuPy installed — using GPU acceleration")
        except ImportError:
            print("⚠ CuPy not available — falling back to CPU (NumPy/SciPy)")
else:
    print("Running locally (not Colab)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from fabemd import FABEMD, _to_xp, _to_numpy, HAS_CUPY

if HAS_CUPY:
    import cupy as cp
    print("Backend: CuPy (GPU)")
else:
    cp = np
    print("Backend: NumPy (CPU)")

## 1. Generate synthetic test images

We create three test images with known frequency content to validate the decomposition:
- **Multi-frequency sinusoidal** — superposition of different spatial frequencies
- **Gaussian blobs + noise** — structured pattern with additive noise
- **Natural-like texture** — combination of gradients, patterns, and noise

In [ ]:
def make_test_images(size: int = 256):
    """Generate three synthetic test images of shape (size, size)."""
    x = np.linspace(0, 1, size)
    y = np.linspace(0, 1, size)
    X, Y = np.meshgrid(x, y)

    # --- Test image 1: multi-frequency sinusoidal ---
    high_freq = np.sin(2 * np.pi * 16 * X) * np.cos(2 * np.pi * 16 * Y)
    mid_freq = np.sin(2 * np.pi * 4 * X + np.pi / 4) * np.cos(2 * np.pi * 4 * Y)
    low_freq = np.sin(2 * np.pi * 1 * X) + np.cos(2 * np.pi * 1 * Y)
    img_sinusoidal = 0.3 * high_freq + 0.4 * mid_freq + 0.3 * low_freq

    # --- Test image 2: Gaussian blobs + noise ---
    blob1 = np.exp(-((X - 0.3) ** 2 + (Y - 0.3) ** 2) / (2 * 0.05 ** 2))
    blob2 = np.exp(-((X - 0.7) ** 2 + (Y - 0.6) ** 2) / (2 * 0.08 ** 2))
    blob3 = np.exp(-((X - 0.5) ** 2 + (Y - 0.8) ** 2) / (2 * 0.04 ** 2))
    rng = np.random.default_rng(42)
    noise = 0.1 * rng.standard_normal((size, size))
    img_blobs = blob1 + 0.8 * blob2 + 0.6 * blob3 + noise

    # --- Test image 3: natural-like texture ---
    gradient = X * 0.5 + Y * 0.5
    texture = (
        np.sin(2 * np.pi * 8 * X) * np.sin(2 * np.pi * 6 * Y) * 0.3
        + np.sin(2 * np.pi * 2 * (X + Y)) * 0.4
    )
    fine_noise = 0.05 * rng.standard_normal((size, size))
    img_texture = gradient + texture + fine_noise

    return {
        "Multi-freq sinusoidal": img_sinusoidal,
        "Gaussian blobs + noise": img_blobs,
        "Natural-like texture": img_texture,
    }


test_images = make_test_images(256)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, img) in zip(axes, test_images.items()):
    im = ax.imshow(img, cmap="gray")
    ax.set_title(name)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("Synthetic Test Images", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Run FABEMD decomposition

We decompose each test image using FABEMD with adaptive window sizing.
The algorithm automatically determines the optimal filter window at each sifting step based on the spatial distribution of local extrema (Bhuiyan et al. §III-B).

In [ ]:
# Instantiate FABEMD with default adaptive window sizing
fabemd = FABEMD(
    max_sift_iterations=10,
    sd_threshold=0.2,
    min_extrema=5,
    max_bimfs=20,
)

# Store decomposition results
results = {}

for name, img in test_images.items():
    print(f"\n{'='*60}")
    print(f"Decomposing: {name}  (shape={img.shape})")
    print(f"{'='*60}")
    gpu_img = _to_xp(img)
    bimfs = fabemd.decompose(gpu_img)
    results[name] = bimfs
    print(f"  → Extracted {len(bimfs)} BIMFs")

## 3. Visualise BIMFs

Each row shows one test image and its extracted BIMFs (highest → lowest frequency), plus the residual.

In [ ]:
for name, bimfs in results.items():
    n = len(bimfs)
    # Compute residual
    original = _to_xp(test_images[name]).astype(np.float64)
    residual = _to_numpy(original - sum(bimfs))

    ncols = min(n + 2, 8)  # original + bimfs + residual, max 8 per row
    nrows = int(np.ceil((n + 2) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    axes = np.atleast_2d(axes)

    all_axes = axes.ravel()
    # Plot original
    all_axes[0].imshow(test_images[name], cmap="gray")
    all_axes[0].set_title("Original", fontsize=9)
    all_axes[0].axis("off")

    # Plot BIMFs
    for i, bimf in enumerate(bimfs):
        ax = all_axes[i + 1]
        ax.imshow(_to_numpy(bimf), cmap="gray")
        ax.set_title(f"BIMF {i + 1}", fontsize=9)
        ax.axis("off")

    # Plot residual
    all_axes[n + 1].imshow(residual, cmap="gray")
    all_axes[n + 1].set_title("Residual", fontsize=9)
    all_axes[n + 1].axis("off")

    # Hide unused axes
    for j in range(n + 2, len(all_axes)):
        all_axes[j].axis("off")

    fig.suptitle(f"FABEMD Decomposition — {name}", fontsize=13)
    plt.tight_layout()
    plt.show()

## 4. Calculate entropy for all BIMFs

Shannon entropy measures the information content / complexity of each BIMF.  
Higher entropy → more randomness; lower entropy → more structured/smooth content.

$$H = -\sum_{i} p_i \ln(p_i)$$

In [ ]:
print(f"{'Image':<30} {'BIMF':>6} {'Entropy':>12} {'Energy':>14}")
print("=" * 66)

entropy_data = {}

for name, bimfs in results.items():
    entropies = FABEMD.calculate_all_entropies(bimfs, bins=256)
    energies = FABEMD.calculate_energies(bimfs)
    entropy_data[name] = entropies

    for i, (ent, eng) in enumerate(zip(entropies, energies)):
        label = name if i == 0 else ""
        print(f"{label:<30} {i + 1:>6} {ent:>12.4f} {eng:>14.4f}")
    print("-" * 66)

In [ ]:
# Bar chart of BIMF entropies per test image
fig, axes = plt.subplots(1, len(entropy_data), figsize=(5 * len(entropy_data), 4))
if len(entropy_data) == 1:
    axes = [axes]

for ax, (name, entropies) in zip(axes, entropy_data.items()):
    indices = np.arange(1, len(entropies) + 1)
    bars = ax.bar(indices, entropies, color=plt.cm.viridis(np.linspace(0.2, 0.8, len(entropies))))
    ax.set_xlabel("BIMF index")
    ax.set_ylabel("Shannon Entropy (nats)")
    ax.set_title(name, fontsize=10)
    ax.set_xticks(indices)

plt.suptitle("BIMF Entropy Distribution", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Reconstruction verification

Verify that the sum of all BIMFs + residual reconstructs the original image (up to floating-point precision).

In [ ]:
xp = cp  # cupy if available, else numpy (aliased above)

print("Reconstruction error (max absolute difference):")
print("-" * 50)
for name, bimfs in results.items():
    original = _to_xp(test_images[name]).astype(np.float64)
    reconstructed = sum(bimfs)
    residual = original - reconstructed
    # Full reconstruction = sum(bimfs) + residual = original  →  error should be ~0
    max_err = float(xp.max(xp.abs(residual)))
    mean_err = float(xp.mean(xp.abs(residual)))
    print(f"  {name:<30}  max={max_err:.2e}  mean={mean_err:.2e}")

## 6. Pipeline integration test

Demonstrate that `FABEMD` can be used as a drop-in replacement for `BEMD` inside `ImageProcessingPipeline`.

In [ ]:
try:
    from image_pipeline import ImageProcessingPipeline, PipelineConfig, BEMD

    # --- Create a config that uses FABEMD ---
    config = PipelineConfig(
        decomposition_method="fabemd",       # ← use FABEMD instead of BEMD
        fabemd_max_sift_iterations=10,
        fabemd_sd_threshold=0.2,
        fabemd_min_extrema=5,
        fabemd_max_bimfs=20,
    )

    pipeline = ImageProcessingPipeline(config)

    # Use the multi-freq sinusoidal test image
    test_img = test_images["Multi-freq sinusoidal"]
    # Normalise to uint16 range for pipeline compatibility
    test_img_u16 = ((test_img - test_img.min()) / (test_img.max() - test_img.min()) * 65535).astype(np.uint16)

    # Decompose using the pipeline's method (which now routes to FABEMD)
    bimfs_pipeline, energies_pipeline = pipeline.decompose_image(test_img_u16, method="fabemd")
    print(f"\nPipeline (FABEMD): extracted {len(bimfs_pipeline)} BIMFs")
    print(f"Energies: {[f'{e:.2f}' for e in energies_pipeline]}")

    # Compare with BEMD
    bimfs_bemd, energies_bemd = pipeline.decompose_image(test_img_u16, method="bemd")
    print(f"\nPipeline (BEMD):   extracted {len(bimfs_bemd)} BIMFs")
    print(f"Energies: {[f'{e:.2f}' for e in energies_bemd]}")

except ImportError as e:
    print(f"⚠ image_pipeline not fully available ({e})")
    print("  Skipping pipeline integration test — FABEMD standalone works fine.")

## 7. FABEMD with fixed vs adaptive window size

Compare the effect of using a fixed window size versus the paper's adaptive window sizing.

In [ ]:
import time

test_img_gpu = _to_xp(test_images["Multi-freq sinusoidal"])

# --- Adaptive window ---
fabemd_adaptive = FABEMD(max_sift_iterations=10, sd_threshold=0.2, max_bimfs=20)
t0 = time.perf_counter()
bimfs_adaptive = fabemd_adaptive.decompose(test_img_gpu)
t_adaptive = time.perf_counter() - t0

# --- Fixed window (size=32, similar to existing BEMD config) ---
fabemd_fixed = FABEMD(max_sift_iterations=10, sd_threshold=0.2, max_bimfs=20, initial_window_size=32)
t0 = time.perf_counter()
bimfs_fixed = fabemd_fixed.decompose(test_img_gpu)
t_fixed = time.perf_counter() - t0

print(f"\nAdaptive window:  {len(bimfs_adaptive)} BIMFs in {t_adaptive:.2f}s")
print(f"Fixed window=32:  {len(bimfs_fixed)} BIMFs in {t_fixed:.2f}s")

# Compare entropies
ent_adaptive = FABEMD.calculate_all_entropies(bimfs_adaptive)
ent_fixed = FABEMD.calculate_all_entropies(bimfs_fixed)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(1, len(ent_adaptive) + 1), ent_adaptive, color="steelblue")
ax1.set_title(f"Adaptive Window ({len(bimfs_adaptive)} BIMFs, {t_adaptive:.2f}s)")
ax1.set_xlabel("BIMF index")
ax1.set_ylabel("Entropy (nats)")

ax2.bar(range(1, len(ent_fixed) + 1), ent_fixed, color="coral")
ax2.set_title(f"Fixed Window=32 ({len(bimfs_fixed)} BIMFs, {t_fixed:.2f}s)")
ax2.set_xlabel("BIMF index")
ax2.set_ylabel("Entropy (nats)")

plt.suptitle("Adaptive vs Fixed Window — Entropy Comparison", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()